# ParFuMor (version YAML & Objets)

- Ajout d'un %store pour la gestion des numéros de kanoniks (16/04/20)

- Attention les règles qui ne changent pas le radical donnent lieu à des mauvais découpages des formes...

In [118]:
# -*- coding: utf8 -*-
import os
from os.path import expanduser
import itertools
import yaml,YamlDuplicates
from yaml.constructor import ConstructorError
import warnings
import ParFuMor as PFM
from ParFuMor import *
import pickle
from IPython.display import HTML, display
#import cellbell 

In [119]:
def ding():
    os.system('afplay /System/Library/Sounds/Submarine.aiff')

# Gestion partagée des numéros à traiter

le block suivant permet de partager les numéros à traiter entre les différents Notebooks.
- %store -r variable lit la variable dans le stock
- %store variable stocke la variable

In [120]:
%store -r numerosKalaba typeKalaba 
%store -r anneeKalaba

# anneeKalaba=25
numerosKalaba=[1,2,3,4,5]
numerosKalaba=[6]
print numerosKalaba
typeKalaba="Kalaba"
%store numerosKalaba 
%store anneeKalaba
%store typeKalaba

[6]
Stored 'numerosKalaba' (list)
Stored 'anneeKalaba' (int)
Stored 'typeKalaba' (str)


In [121]:
home = expanduser("~")
repertoire=home+"/sDrive/Cours/Bordeaux/L1-LinguistiqueGenerale/00-ProjetKalaba/"
annee=anneeKalaba
if typeKalaba!="Kanonik":
    serie=repertoire+"%d-"%annee
    nomsKalabas=[serie+"K%d/"%num for num in numerosKalaba]
else:
    serie=repertoire+"%d-Kanoniks/"%annee
    nomsKalabas=[serie+"Kanonik-%02d/"%num for num in numerosKalaba]

# Traitement

In [122]:
def getExemple(lDict):
    exKey=lDict.keys()[0]
    if isinstance(lDict[exKey],dict):
        result=getExemple(lDict[exKey])
    elif isinstance(lDict[exKey],list):
        result=lDict[exKey][0]
    return result

In [123]:
PFM.duplicateErrors=[]
for serie in nomsKalabas:
    print
    print serie

    with open(serie+"Gloses.yaml", 'r') as stream:
        gloses=yaml.safe_load(stream)
        PFM.gloses=gloses
    with open(serie+"Stems.yaml", 'r') as stream:
        try:
            stems=yaml.safe_load(stream)
            PFM.stems=stems
        except ConstructorError,msg:
            print msg
            continue
    
    lexiqueTestPrep=getExemple(stems["PREP"]).lower()
    lexiqueTestNoun=getExemple(stems["NOM"]).lower()
    lexiqueTestHyper=getExemple(stems["ADJ"]).lower()
    
    with open(serie+"Blocks.yaml", 'r') as stream:
        blocks=yaml.safe_load(stream)
        PFM.blocks=blocks
    with open(serie+"Phonology.yaml", 'r') as stream:
        phonology=yaml.safe_load(stream)
        PFM.phonology=phonology
    with open(serie+"MorphoSyntax.yaml", 'r') as stream:
        morphosyntax=yaml.safe_load(stream)
        PFM.morphosyntax=morphosyntax

        
    for cat in morphosyntax["Attributs"]:
        if sorted(morphosyntax["Attributs"][cat])!=sorted(gloses[cat].keys()):
            warnings.warn("\n"+serie+"\nLes attributs de %s ne sont pas cohérents\nMorphosyntax => %s\nGloses => %s"%(cat,", ".join(morphosyntax["Attributs"][cat]),", ".join(gloses[cat].keys())))
    
    regles=Regles()
    PFM.regles=regles
    for categorie in blocks:
        regles.addBlocs(categorie,blocks[categorie])


    paradigmes=Paradigmes()
    PFM.paradigmes=paradigmes

    hierarchieCF=HierarchieCF()
    PFM.hierarchieCF=hierarchieCF
    lexique=Lexique()
    PFM.lexique=lexique


    for cat in gloses:
    #    print cat
        attributes=[]
        if gloses[cat]:
            if set(gloses[cat].keys())==set(morphosyntax["Attributs"][cat]):
                features=morphosyntax["Attributs"][cat]
    #            print "inhérent",features
            else:
                features=gloses[cat].keys()
    #            print "contextuel",features
            for attribute in features:
                attributes.append(gloses[cat][attribute])
            nuplets=(itertools.product(*attributes))
            for nuplet in nuplets:
                proprietes=[cat]
                for element in range(len(nuplet)):
                    proprietes.append(u"%s=%s"%(features[element],nuplet[element]))
                paradigmes.addForme(cat,proprietes)

    analyserGloses(gloses)
    analyserStems(stems)
        
    with open(serie+"Hierarchie-S2.pkl", 'wb') as output:
       pickle.dump(hierarchieCF, output, pickle.HIGHEST_PROTOCOL)
    with open(serie+"Lexique-S2.pkl", 'wb') as output:
       pickle.dump(lexique, output, pickle.HIGHEST_PROTOCOL)
    with open(serie+"Regles-S2.pkl", 'wb') as output:
       pickle.dump(regles, output, pickle.HIGHEST_PROTOCOL)


    # PFM.lexique.lexemes[lexiqueTestPrep]
    if lexiqueTestPrep in PFM.lexique.lexemes:
        PFM.lexique.lexemes[lexiqueTestPrep]
    else:
        PFM.lexique.lexemes[getExemple(stems["PREP"])]

    mot=lexiqueTestNoun
    # classesNom=PFM.lexique.formeLexeme[mot.lower()][0].split('.')[1:]
    # [classeElement for classeElement in classesNom if classeElement in gloses["NOM"]["Genre"]]

    PFM.lexique.formeLexeme[lexiqueTestHyper][0]

if PFM.duplicateErrors:
    print
    print "=======ERREURS========"
    print
    for ligne in PFM.duplicateErrors:
        print ligne



/Users/gilles/sDrive/Cours/Bordeaux/L1-LinguistiqueGenerale/00-ProjetKalaba/25-K6/
NOM : Genre Nombre Cas
VER : Trans Temps Pers Genre
ADJ : Genre Nombre
PRO : Genre Nombre Cas
DET : Genre Nombre Cas
head stems
head stems,NOM
head stems,NOM,Inan
head stems,NOM,Inan,Boisson
head stems,NOM,Inan,Lieu
head stems,NOM,Inan,Lieu,Campagne
head stems,NOM,Inan,Lieu,Ville
head stems,NOM,Anim
head stems,NOM,Anim,Hum
head stems,NOM,Anim,NonHum
head stems,NOM,Anim,NonHum,N1
head stems,NOM,Anim,NonHum,N2
head stems,VER
head stems,VER,VI
head stems,VER,VTRANS
head stems,VER,VTRANS,VT
head stems,VER,VTRANS,VD
head stems,ADJ


### Bilan

In [124]:
print PFM.lexique.formeLexeme[lexiqueTestHyper][0]
# print PFM.lexique.formeLexeme[lexiqueTestNoun][0]
print PFM.lexique.formeLexeme[lexiqueTestPrep][0]

courageux.A1
de


### Summary

In [125]:
case="CF=N3"
m=re.match(ur"(.*=)(.*)","CF=N1|N2")
if m:
    altTrait=m.group(1)
    altValeurs=m.group(2).split("|")
    print altTrait
    altTraits=[altTrait+v for v in altValeurs]
    print altTraits
    if any(t == case for t in altTraits):
        print "applies"
    else:
        print "does not apply"

CF=
['CF=N1', 'CF=N2']
does not apply


In [126]:
regles.getRules("VER",'CF=V1, Temps=PRS, Pers=3Pl')
regles.getRules("IND","Nombre=Sg, Genre=Inan")

[]

In [127]:
cat="N"
triLex={}
for element in PFM.lexique.lexemes:
#    print element
    elementElts=element.split(".")
    nom=elementElts[0]
    if len(elementElts)==3:
        if elementElts[1].startswith(cat):
            if not elementElts[2] in triLex:
                triLex[elementElts[2]]=set()
            triLex[elementElts[2]].add((PFM.lexique.lexemes[element].stem,nom))            
triLex

{}

In [128]:
for l in PFM.lexique.lexemes:
    print l,PFM.lexique.lexemes[l].stem

découvrir.VTRANS.VT.V2 Duvor
PRO r
entrer.VI.V1 bowom
crapaud.Anim.NonHum.N1.N11 TotoT
furieux.A1 wuvuN
DEM k
serpent.Anim.NonHum.N1.N12 goweD
café.Inan.Boisson.N2 Dovuj
DEF p
créature.Anim.NonHum.N2.N21 damow
courageux.A1 Nopun
bras.Anim.Hum.N1 TotoT
tourner-Mov.VI.V2 zoSun
sur zow
sorcier.Anim.Hum.N2 Suvet
lit.Inan.Lieu.Ville.N1 DebuN
gargouille.Anim.NonHum.N3 ZazuT
grand.A2 bubej
dragon.Anim.NonHum.N3 Zunup
à zin
de juvi
tornade.Anim.NonHum.N2.N22 guSug
placer.VTRANS.VT.V2 Novib
lever-N.Inan.Boisson.N2 NiwaT
jardiner.VI.V2 wubuS
protéger.VTRANS.VT.V1 fubew
chasser-Act.VTRANS.VT.V1 fuTaS
donner.VTRANS.VD.V1 videt
méchant.A2 zozur
côté.Inan.Lieu.Campagne.N2 Tezad
pour wunoj
combattant.Anim.Hum.N1 NuDiZ
main.Anim.Hum.N2 Zujaw
IND r
manquer-Med.VI.V1 Sugut


# Conclusion

In [129]:
ding()

In [141]:
print hierarchieCF.classes
print
print hierarchieCF.superieur
print
print hierarchieCF.categorie
print
print hierarchieCF.trait
print
print hierarchieCF.sets
print
print hierarchieCF.inherents

{'NOM': ['Inan', 'Anim'], 'VI': ['V1', 'V2'], 'Anim': ['Hum', 'NonHum'], 'VER': ['VI', 'VTRANS'], 'Ville': ['N1'], 'Hum': ['N1', 'N2'], 'NonHum': ['N1', 'N2', 'N3'], 'VD': ['V1'], 'VTRANS': ['VT', 'VD'], 'Inan': ['Boisson', 'Lieu'], 'Campagne': ['N2'], 'VT': ['V1', 'V2'], 'Lieu': ['Campagne', 'Ville'], 'Boisson': ['N2'], 'N1': ['N12', 'N11'], 'N2': ['N22', 'N21'], 'ADJ': ['A1', 'A2']}

{'Inan': 'NOM', 'Anim': 'NOM', 'Hum': 'Anim', 'A1': 'ADJ', 'V1': 'VD', 'V2': 'VT', 'A2': 'ADJ', 'NonHum': 'Anim', 'Boisson': 'Inan', 'N22': 'N2', 'N21': 'N2', 'VTRANS': 'VER', 'VD': 'VTRANS', 'VI': 'VER', 'VT': 'VTRANS', 'Campagne': 'Lieu', 'N1': 'NonHum', 'N2': 'NonHum', 'N3': 'NonHum', 'N12': 'N1', 'Ville': 'Lieu', 'N11': 'N1', 'Lieu': 'Inan'}

{'Inan': 'NOM', 'Anim': 'NOM', 'Hum': 'NOM', 'A1': 'ADJ', 'V1': 'VER', 'V2': 'VER', 'A2': 'ADJ', 'NonHum': 'NOM', 'Boisson': 'NOM', 'N22': 'NOM', 'N21': 'NOM', 'VTRANS': 'VER', 'VD': 'VER', 'VI': 'VER', 'VT': 'VER', 'Campagne': 'NOM', 'N1': 'NOM', 'N2': 'NOM', '

In [131]:
def makeHierarchy(hCF,f,sup=hierarchieCF.superieur):
    nH=[k for d in hCF for k,v in d.items() if v==f]
    print(nH)
    hF={}
    for k,v in sup.items():
        if k in nH:
            hF[k]=v
    return hF

def getHierarchy(c,hF):
    result=[c]
    k=c
    while k in hF and hF[k] in hF:
        result.append(hF[k])
        k=hF[k]
    return result

In [133]:
h=makeHierarchy(hierarchieCF.trait["VER"],"Trans")
getHierarchy("VD",h)

['VI', 'VTRANS', 'VT', 'VD']


['VD', 'VTRANS']

In [77]:
genres=[k for d in hierarchieCF.trait["NOM"] for k,v in d.items() if v=="Genre"]
nClasses=[k for d in hierarchieCF.trait["NOM"] for k,v in d.items() if v=="CF"]
genres

['Inan', 'Anim', 'Boisson', 'Lieu', 'Campagne', 'Ville', 'Hum', 'NonHum']

In [62]:
hGenres={}
for k,v in hierarchieCF.superieur.items():
    if k in genres and v in genres:
        hGenres[k]=v
hGenres

({'Boisson': 'Inan',
  'Campagne': 'Lieu',
  'Hum': 'Anim',
  'Lieu': 'Inan',
  'NonHum': 'Anim',
  'Ville': 'Lieu'},
 {'Boisson': ['Boisson', 'Inan'],
  'Campagne': ['Campagne', 'Lieu', 'Inan'],
  'Hum': ['Hum', 'Anim'],
  'Lieu': ['Lieu', 'Inan'],
  'NonHum': ['NonHum', 'Anim'],
  'Ville': ['Ville', 'Lieu', 'Inan']})

In [64]:
hClasses={}
for k,v in hierarchieCF.superieur.items():
    if k in nClasses and v in nClasses:
        hClasses[k]=v
hClasses

{'N11': 'N1', 'N12': 'N1', 'N21': 'N2', 'N22': 'N2'}

In [71]:
def getGenres(genre):
    result=[genre]
    k=genre
    while k in hGenres:
        result.append(hGenres[k])
        k=hGenres[k]
    return result

def getCFs(cf):
    result=[cf]
    k=cf
    while k in hClasses:
        result.append(hClasses[k])
        k=hClasses[k]
    return result

In [72]:
getGenres("Hum"),getCFs("N12")


(['Hum', 'Anim'], ['N12', 'N1'])

In [59]:
hierarchieCF.trait["VER"]

[{'VI': 'CF'}, {'VT': 'CF'}, {'V1': 'CF'}, {'V2': 'CF'}]